# 02 - Prétraitement & extraction d'embeddings (ResNet pré-entraîné)
**Branche** : `feature/02-feature-extraction`

Backbone **gelé** ; images en niveaux de gris répliquées sur 3 canaux, redimensionnées à 224×224, normalisées ImageNet. Pas d'augmentation à l'extraction.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from brainscanai.config import load_config, resolve_path
from brainscanai.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
PROC, FEAT = resolve_path(cfg, "processed", create=True), resolve_path(cfg, "features", create=True)
FIG, TAB = resolve_path(cfg, "figures", create=True), resolve_path(cfg, "tables", create=True)
from brainscanai.features.extractor import FeatureExtractor
from brainscanai.features.io import save_features, load_features
from brainscanai.clustering.reduce import standardize, pca_reduce

In [ ]:
manifest = pd.read_csv(PROC / 'manifest_clean.csv')
fc = cfg['features']
# Passez pretrained=False pour tester hors-ligne (embeddings non pertinents)
ext = FeatureExtractor(fc['backbone'], pretrained=fc['pretrained'], layer=fc['layer'], device=fc['device'])
print('device :', ext.device)

In [ ]:
# Astuce : tester d'abord sur un échantillon (ex. manifest.sample(64)) avant le jeu complet
X = ext.extract(manifest.path, cfg['data']['image_size'], fc['batch_size'], fc['num_workers'])
print(X.shape, X.dtype)

## Contrôles de sortie (exploitabilité)

In [ ]:
assert np.isfinite(X).all(), 'NaN/Inf dans les embeddings'
print('dimensions constantes :', int((X.std(0) < 1e-8).sum()))
Xs, _ = standardize(X)
Z, pca = pca_reduce(Xs, cfg['clustering']['pca_components'], cfg['seed'])
print(f'PCA : {X.shape[1]} -> {Z.shape[1]} dims ({pca.explained_variance_ratio_.sum():.1%} de variance)')
plt.plot(np.cumsum(pca.explained_variance_ratio_)); plt.xlabel('composantes'); plt.ylabel('variance cumulée'); plt.show()

## Comparaison de couches d'extraction (optionnel)
Une couche intermédiaire (`layer3`) capte parfois mieux les textures que la dernière couche.

In [ ]:
# for layer in ['layer3', 'avgpool']:
#     e = FeatureExtractor(fc['backbone'], True, layer, fc['device'])
#     Xl = e.extract(manifest.path); print(layer, Xl.shape)  # puis comparer l'ARI (notebook 03)

In [ ]:
path = save_features(manifest, X, FEAT)
Xb, mb = load_features(path, manifest)   # relecture = vérification d'alignement
assert (mb.image_id.values == manifest.image_id.values).all()
path